In [4]:
import asyncio
import re
from typing import List, Dict
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Constants (adjust as needed)

COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M",
    "EBOOK": "N",
    "COURSES": "O",
    "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q",
    "WEBINAR": "R",
    "SERVICES": "S",
    "PODCAST": "T",
    "SHOP": "U"
}
EXTRACTION_METADATA_COLUMN = "V"

# Content delimiter - making it more flexible
CONTENT_DELIMITER = "-----NEXT CONTENT FROM HERE-----"

class GoogleSheetsManager:
    def __init__(self, credentials_file: str):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """
        Extracts the spreadsheet ID from a Google Sheet URL.
        """
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")

def parse_content_pieces(content: str, expected_count: int, category: str, row_num: int) -> List[str]:
    """
    Enhanced content parsing with better error handling and debugging.
    
    Args:
        content (str): The raw content from the cell
        expected_count (int): Number of pieces expected based on metadata
        category (str): Category name for debugging
        row_num (int): Row number for debugging
    
    Returns:
        List[str]: List of parsed content pieces
    """
    if not content or not content.strip():
        print(f"Warning: Empty content in row {row_num}, category {category}")
        return []
    
    # Clean up the content - remove extra whitespace and normalize line endings
    cleaned_content = content.strip()
    
    # Debug: Show the first part of the content
    print(f"Debug - Row {row_num}, {category}: Content preview: '{cleaned_content[:100]}...'")
    
    # Try different variations of the delimiter in case there are formatting issues
    delimiters_to_try = [
        CONTENT_DELIMITER,
        CONTENT_DELIMITER.strip(),
        "-----NEXT CONTENT FROM HERE-----",
        "--- --NEXT CONTENT FROM HERE-----",  # Common typo
        "-----NEXT CONTENT FROM HERE--- --",  # Common typo
    ]
    
    pieces = None
    delimiter_used = None
    
    # Try each delimiter variation
    for delimiter in delimiters_to_try:
        if delimiter in cleaned_content:
            pieces = cleaned_content.split(delimiter)
            delimiter_used = delimiter
            break
    
    # If no delimiter found, treat as single piece
    if pieces is None:
        pieces = [cleaned_content]
        print(f"Debug - Row {row_num}, {category}: No delimiter found, treating as single piece")
    else:
        print(f"Debug - Row {row_num}, {category}: Found {len(pieces)} pieces using delimiter '{delimiter_used}'")
    
    # Clean up each piece (remove leading/trailing whitespace)
    cleaned_pieces = [piece.strip() for piece in pieces if piece.strip()]
    
    # Handle expected count validation
    if expected_count == 1 and len(cleaned_pieces) == 1:
        return cleaned_pieces
    elif expected_count > 1:
        if len(cleaned_pieces) >= expected_count:
            return cleaned_pieces[:expected_count]
        else:
            print(f"Warning: Expected {expected_count} pieces but found {len(cleaned_pieces)} in row {row_num}, category {category}")
            print(f"  Available pieces: {[p[:50] + '...' if len(p) > 50 else p for p in cleaned_pieces]}")
            return cleaned_pieces
    else:
        # Expected count is 0 or negative - shouldn't happen but handle gracefully
        print(f"Warning: Unexpected expected_count {expected_count} in row {row_num}, category {category}")
        return cleaned_pieces

async def read_data_from_sheet(spreadsheet_id: str, sheet_mgr: GoogleSheetsManager, categories: List[str]) -> List[Dict[str, List[str]]]:
    """
    Efficiently reads data from a Google Sheet based on extraction metadata and category columns.
    """
    # Validate category columns
    category_columns = {cat: COLUMN_TO_WRITE_URL_TO.get(cat.upper()) for cat in categories}
    if any(col is None for col in category_columns.values()):
        raise ValueError("One or more categories do not have a defined column in COLUMN_TO_WRITE_URL_TO")

    # Set metadata column
    metadata_column = EXTRACTION_METADATA_COLUMN

    # Prepare ranges for batch reading (start from row 2)
    ranges = [f"{col}2:{col}" for col in category_columns.values()] + [f"{metadata_column}2:{metadata_column}"]

    # Batch read data asynchronously
    try:
        response = await asyncio.to_thread(
            sheet_mgr.service.spreadsheets().values().batchGet(spreadsheetId=spreadsheet_id, ranges=ranges).execute
        )
    except Exception as e:
        raise RuntimeError(f"Failed to read data from spreadsheet {spreadsheet_id}: {str(e)}")

    value_ranges = response.get('valueRanges', [])

    # Extract column data
    column_data = {}
    for i, col in enumerate(category_columns.values()):
        column_data[col] = value_ranges[i].get('values', [])
    column_data[metadata_column] = value_ranges[-1].get('values', [])

    # Determine the number of rows
    num_rows = max(len(values) for values in column_data.values()) if column_data else 0

    # Process each row
    data = []
    for i in range(num_rows):
        row_data = {}
        # Read metadata
        metadata_value = column_data[metadata_column][i][0] if i < len(column_data[metadata_column]) and column_data[metadata_column][i] else ''
        print(f"Row {i+2} metadata: {metadata_value}")

        # Parse metadata
        category_n_dict = {}
        for part in metadata_value.split(','):
            if '=' in part:
                cat, n_str = part.split('=', 1)
                try:
                    n = int(n_str.strip())
                    category_n_dict[cat.strip().upper()] = n
                except ValueError:
                    print(f"Warning: Invalid metadata '{part}' in row {i+2}")

        # Process each category
        for category in categories:
            category_upper = category.upper()
            if category_upper in category_n_dict and category_n_dict[category_upper] > 0:
                col = category_columns[category]
                content = column_data[col][i][0] if i < len(column_data[col]) and column_data[col][i] else ''
                expected_count = category_n_dict[category_upper]
                
                # Use the enhanced parsing function
                parsed_pieces = parse_content_pieces(content, expected_count, category, i+2)
                if parsed_pieces:
                    row_data[category] = parsed_pieces

        if not row_data:
            print(f"Row {i+2} has no data (empty row_data)")
        data.append(row_data)

    return data

# Example usage
async def main():
    # Replace with your actual values
    SHEET_URL = "https://docs.google.com/spreadsheets/d/1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc/edit?gid=0#gid=0"
    CREDENTIALS_FILE = "url-to-email-445616-cebe4868914f.json"
    CATEGORIES = ["ABOUT_US","EBOOK","COURSES", "RECENT_BLOG", "TESTIMONIALS", "WEBINAR", "SERVICES", "PODCAST", "SHOP" ]

    try:
        # Initialize GoogleSheetsManager
        sheet_mgr = GoogleSheetsManager(CREDENTIALS_FILE)
        # Extract spreadsheet ID
        spreadsheet_id = sheet_mgr.extract_spreadsheet_id(SHEET_URL)
        print(f"Extracted spreadsheet ID: {spreadsheet_id}")
        # Read data
        data = await read_data_from_sheet(spreadsheet_id, sheet_mgr, CATEGORIES)
        # Print results for verification
        for row_idx, row_data in enumerate(data, start=2):
            print(f"\nRow {row_idx}:")
            for category, contents in row_data.items():
                print(f"  {category}: Found {len(contents)} pieces")
                for idx, content in enumerate(contents, start=1):
                    # Show more of the content for debugging
                    preview = content[:100] + "..." if len(content) > 100 else content
                    print(f"    Content {idx}: {preview}")
        return data
    except Exception as e:
        print(f"Error: {str(e)}")
        return []
# For Jupyter or async environments, run:
data = await main()

Extracted spreadsheet ID: 1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc
Row 2 metadata: 
Row 2 has no data (empty row_data)
Row 3 metadata: PODCAST=0,ABOUT_US=3,SHOP=0,RECENT_BLOG=0,EBOOK=0,COURSES=0,TESTIMONIALS=1,WEBINAR=0,SERVICES=4
Debug - Row 3, ABOUT_US: Content preview: 'ABOUT US
Our
mission...
We stop burnout, disengagement, conflict, and violence in the workplace usin...'
Debug - Row 3, ABOUT_US: Found 3 pieces using delimiter '-----NEXT CONTENT FROM HERE-----'
Debug - Row 3, TESTIMONIALS: Content preview: 'All Posts
Search
Read a Participant Testimonial - September 2024
Zach Stone
Oct 1, 2024
1 min read
T...'
Debug - Row 3, TESTIMONIALS: No delimiter found, treating as single piece
Debug - Row 3, SERVICES: Content preview: 'For Executives
Use systems thinking and policy shifts to build resilient and performant work communi...'
Debug - Row 3, SERVICES: Found 4 pieces using delimiter '-----NEXT CONTENT FROM HERE-----'
Row 4 metadata: PODCAST=2,ABOUT_US=10,SHOP=3,RECENT_BLOG=3,EB

In [7]:
def explore_specific_content(data, row_number, category, piece_number=1):
    """
    Show the full content of a specific piece for detailed examination
    """
    try:
        row_data = data[row_number - 2]  # Adjust for 0-based indexing
        if category in row_data:
            pieces = row_data[category]
            if piece_number <= len(pieces):
                content = pieces[piece_number - 1]
                print(f"Full content for Row {row_number}, {category}, Piece {piece_number}:")
                print("-" * 60)
                print(content)
                print("-" * 60)
                print(f"Content length: {len(content)} characters")
                print(f"Line count: {len(content.split(chr(10)))}")
            else:
                print(f"Piece {piece_number} not found. Available pieces: 1-{len(pieces)}")
        else:
            print(f"Category {category} not found in row {row_number}")
            print(f"Available categories: {list(row_data.keys())}")
    except IndexError:
        print(f"Row {row_number} not found. Available rows: 2-{len(data) + 1}")

explore_specific_content(data, 3, "ABOUT_US", 1) 

Full content for Row 3, ABOUT_US, Piece 1:
------------------------------------------------------------
ABOUT US
Our
mission...
We stop burnout, disengagement, conflict, and violence in the workplace using techniques built-in war zones for high-stress professionals. Our methods are research-proven to reduce absenteeism, assaults, complaints, rule violations, health-care costs, workplace conflict, and resource wasting. Our innovative consulting and training utilize best practices from behavioral health, violence prevention, martial arts/yoga, and the “post-war reconstruction” model.
​
Our mission is to assist organizations and their workforce in building resilience and staying safe on the job. Our training is interactive, utilizing case studies and roleplay scenarios taken directly from your employees' real-world work experiences. Your workforce will gain practical strategies to de-escalate conflict and crisis when working with difficult populations.
​
Want to know more about how we got

In [5]:
data 

[{},
 {'ABOUT_US': ["ABOUT US\nOur\nmission...\nWe stop burnout, disengagement, conflict, and violence in the workplace using techniques built-in war zones for high-stress professionals. Our methods are research-proven to reduce absenteeism, assaults, complaints, rule violations, health-care costs, workplace conflict, and resource wasting. Our innovative consulting and training utilize best practices from behavioral health, violence prevention, martial arts/yoga, and the “post-war reconstruction” model.\n\u200b\nOur mission is to assist organizations and their workforce in building resilience and staying safe on the job. Our training is interactive, utilizing case studies and roleplay scenarios taken directly from your employees' real-world work experiences. Your workforce will gain practical strategies to de-escalate conflict and crisis when working with difficult populations.\n\u200b\nWant to know more about how we got started?\nRead More\nOur Team\nOur Leadership Team\nCharlotte DiB

In [8]:
def explore_all_content(data, row_number, category):
    """
    Show all content pieces for a specific row and category.
    """
    try:
        row_data = data[row_number - 2]  # Adjust for 0-based indexing
        if category in row_data:
            pieces = row_data[category]
            print(f"\n✅ Full content for Row {row_number}, Category: {category}")
            print("=" * 60)
            for idx, content in enumerate(pieces, start=1):
                print(f"\n--- Piece {idx} ---")
                print(content)
                print("-" * 60)
                print(f"Content length: {len(content)} characters")
                print(f"Line count: {len(content.splitlines())}")
        else:
            print(f"⚠️ Category '{category}' not found in row {row_number}")
            print(f"Available categories: {list(row_data.keys())}")
    except IndexError:
        print(f"⚠️ Row {row_number} not found. Available rows: 2 to {len(data) + 1}")


In [9]:
explore_all_content(data, 3, "ABOUT_US")



✅ Full content for Row 3, Category: ABOUT_US

--- Piece 1 ---
ABOUT US
Our
mission...
We stop burnout, disengagement, conflict, and violence in the workplace using techniques built-in war zones for high-stress professionals. Our methods are research-proven to reduce absenteeism, assaults, complaints, rule violations, health-care costs, workplace conflict, and resource wasting. Our innovative consulting and training utilize best practices from behavioral health, violence prevention, martial arts/yoga, and the “post-war reconstruction” model.
​
Our mission is to assist organizations and their workforce in building resilience and staying safe on the job. Our training is interactive, utilizing case studies and roleplay scenarios taken directly from your employees' real-world work experiences. Your workforce will gain practical strategies to de-escalate conflict and crisis when working with difficult populations.
​
Want to know more about how we got started?
Read More
Our Team
Our Leadersh

In [10]:
def get_content_pieces(data, row_number, category):
    """
    Return a list of all content pieces for a specific row and category.
    """
    try:
        row_data = data[row_number - 2]  # Adjust for 0-based indexing
        if category in row_data:
            return row_data[category]
        else:
            return []  # Category not found in the row
    except IndexError:
        return []  # Row not found


In [11]:
get_content_pieces(data, 3, "ABOUT_US")



["ABOUT US\nOur\nmission...\nWe stop burnout, disengagement, conflict, and violence in the workplace using techniques built-in war zones for high-stress professionals. Our methods are research-proven to reduce absenteeism, assaults, complaints, rule violations, health-care costs, workplace conflict, and resource wasting. Our innovative consulting and training utilize best practices from behavioral health, violence prevention, martial arts/yoga, and the “post-war reconstruction” model.\n\u200b\nOur mission is to assist organizations and their workforce in building resilience and staying safe on the job. Our training is interactive, utilizing case studies and roleplay scenarios taken directly from your employees' real-world work experiences. Your workforce will gain practical strategies to de-escalate conflict and crisis when working with difficult populations.\n\u200b\nWant to know more about how we got started?\nRead More\nOur Team\nOur Leadership Team\nCharlotte DiBartolomeo\nChief Ex